# SpeechKit: синтез и распознавание коротких сообщений

В этом ноутбуке мы знакомимся с двумя базовыми задачами SpeechKit:

* **TTS** (*text-to-speech*) — превращение текста в аудио.
* **STT** (*speech-to-text*) — превращение аудио обратно в текст.

В учебных приложениях эти две возможности часто появляются вместе. Например, агент может принять голосовое сообщение в мессенджере, распознать его, составить ответ с помощью LLM и озвучить ответ пользователю. В этом ноутбуке мы ограничимся короткими аудиофрагментами до 30 секунд: это удобный режим для голосовых заметок, быстрых подсказок, интерактивных тренажёров и простых ботов.

Мы сделаем три вещи:

1. Синтезируем фразу «привет, мир».
2. Соберём маленькую радиопередачу «минутка анекдотов» из нескольких синтезированных фрагментов и разных голосов.
3. Распознаем получившийся аудиофайл обратно в текст.

Полезные ссылки:

* [Документация SpeechKit](https://yandex.cloud/ru/docs/speechkit/)
* [SpeechKit в AI Studio SDK](https://aistudio.yandex.ru/docs/en/ai-studio/sdk-ref/types/speechkit.html)
* [Пример TTS в AI Studio SDK](https://github.com/yandex-cloud/yandex-ai-studio-sdk/blob/master/examples/sync/speechkit/text_to_speech/run_simple.py)
* [Пример STT в AI Studio SDK](https://github.com/yandex-cloud/yandex-ai-studio-sdk/blob/master/examples/sync/speechkit/speech_to_text/run_simple.py)
* [Yandex Cloud AI Studio](https://yandex.cloud/ru/docs/ai-studio/)


In [ ]:
%pip install --quiet yandex-ai-studio-sdk openai python-dotenv


> **ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

## Авторизация

Для работы с Yandex Cloud нам понадобится `folder_id` (идентификатор каталога) и `api_key` (API-ключ сервисного аккаунта). Мы предполагаем, что эти значения хранятся в файле `.env` в текущей директории, который можно скачать, выполнив ячейку ниже (убедитесь предварительно, что в ней указан правильный адрес для скачивания):


In [ ]:
!curl -o .env {{url_of_dotenv_file}}


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio, Markdown, display
from openai import OpenAI
from yandex_ai_studio_sdk import AIStudio

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)

sdk = AIStudio(folder_id=folder_id, auth=api_key)

def printx(string):
    display(Markdown(string))

audio_dir = Path("generated_audio")
audio_dir.mkdir(exist_ok=True)

print(f"✅ Авторизация настроена (folder_id: {folder_id[:8]}...)")


✅ Авторизация настроена (folder_id: b1gbicod...)


## Часть 1. Синтезируем «привет, мир»

Синтез речи начинается с выбора модели TTS и голоса. В SpeechKit голос — это не просто тембр: у разных голосов может отличаться интонация, темп и естественность произношения в разных сценариях. В учебных примерах удобно начинать с короткого текста, чтобы быстро проверить авторизацию, формат аудио и воспроизведение в ноутбуке.

Ниже мы просим SpeechKit вернуть WAV-файл. WAV удобен для обучения: его легко проигрывать в Jupyter, склеивать стандартной библиотекой Python и отправлять обратно в STT.


In [2]:
def synthesize_wav(text: str, path: str | Path, voice: str = "jane", speed: float = 1.0) -> Path:
    """Синтезирует текст в WAV-файл и возвращает путь к файлу."""
    path = Path(path)
    tts = sdk.speechkit.text_to_speech(
        voice=voice,
        audio_format="WAV",
    )
    if hasattr(tts, "configure"):
        tts = tts.configure(speed=speed)
    result = tts.run(text)
    path.write_bytes(result.data)
    return path

hello_path = synthesize_wav("Привет, мир!", audio_dir / "hello_world.wav", voice="jane")
Audio(filename=str(hello_path))


## Часть 2. Радиопередача «Минутка анекдотов»

Теперь сделаем более жизненный пример. В реальном продукте аудио часто собирается из нескольких частей:

* заставка или вступление;
* повторяющаяся служебная фраза;
* основной текст, который может генерироваться моделью;
* разные голоса для разных ролей.

Мы попросим LLM придумать три коротких добрых анекдота на заданные темы.

In [3]:
topics = ["программиста", "робота-помощника", "студента на экзамене"]

prompt = """Придумай три коротких добрых анекдота на русском языке.
Темы: программист, робот-помощник, студент на экзамене.
Верни только JSON-массив объектов вида:
[
  {"topic": "программиста", "text": "..."}
]
Без markdown и без пояснений.
"""

res = client.responses.create(model=model, input=prompt)
printx(res.output_text)


[
  {"topic": "программиста", "text": "Программист уронил телефон в унитаз. Вытащил, просушивает. Говорит: главное — не паниковать, бэкап был."},
  {"topic": "робот-помощника", "text": "Робот-помощник каждое утро делает бутерброд и оставляет записку: 'Съешь меня, а потом — завтрак'."},
  {"topic": "студента на экзамене", "text": "Студент на экзамене говорит: 'Я всё знаю, просто забыл'. Экзаменатор отвечает: 'Зато я всё помню, даже то, чего вы не знаете'. Студент получил тройку с плюсом."}
]

Здесь мы из соображений простоты не используем структурный ответ, хотя стоило бы. Мы же просто выцепим JSON из ответа:

In [4]:
import json

def extract_json_array(text: str):
    start = text.find("[")
    end = text.rfind("]") + 1
    return json.loads(text[start:end])

jokes = extract_json_array(res.output_text)
jokes


[{'topic': 'программиста',
  'text': 'Программист уронил телефон в унитаз. Вытащил, просушивает. Говорит: главное — не паниковать, бэкап был.'},
 {'topic': 'робот-помощника',
  'text': "Робот-помощник каждое утро делает бутерброд и оставляет записку: 'Съешь меня, а потом — завтрак'."},
 {'topic': 'студента на экзамене',
  'text': "Студент на экзамене говорит: 'Я всё знаю, просто забыл'. Экзаменатор отвечает: 'Зато я всё помню, даже то, чего вы не знаете'. Студент получил тройку с плюсом."}]

Теперь склеим из этих анекдотов "радиопередачу". Диктор одним голосом объявляет передачу и тему, а другой голос зачитывает сам анекдот. Так мы одновременно видим, как TTS работает с несколькими голосами и как можно строить маленький аудиосценарий.

In [5]:
import wave

def concat_wav_files(paths: list[Path], output_path: str | Path) -> Path:
    """Склеивает WAV-файлы с одинаковыми параметрами в один WAV."""
    output_path = Path(output_path)
    frames = []
    params = None
    for path in paths:
        with wave.open(str(path), "rb") as wav:
            if params is None:
                params = wav.getparams()
            elif wav.getparams()[:3] != params[:3]:
                raise ValueError(f"Файл {path} имеет другие параметры WAV")
            frames.append(wav.readframes(wav.getnframes()))
    with wave.open(str(output_path), "wb") as out:
        out.setparams(params)
        for chunk in frames:
            out.writeframes(chunk)
    return output_path

parts = []
intro = "Приветствую вас, с вами минутка анекдотов."
parts.append(synthesize_wav(intro, audio_dir / "radio_00_intro.wav", voice="jane"))

script_lines = [intro]
for i, joke in enumerate(jokes, start=1):
    topic = joke["topic"]
    announcement = f"Анекдот про {topic}."
    joke_text = joke["text"]
    parts.append(synthesize_wav(announcement, audio_dir / f"radio_{i:02d}_topic.wav", voice="jane"))
    parts.append(synthesize_wav(joke_text, audio_dir / f"radio_{i:02d}_joke.wav", voice="zahar"))
    script_lines.extend([announcement, joke_text])

radio_path = concat_wav_files(parts, audio_dir / "minute_of_jokes.wav")
print("Сценарий передачи:")
print("\n".join(script_lines))
Audio(filename=str(radio_path))


Сценарий передачи:
Приветствую вас, с вами минутка анекдотов.
Анекдот про программиста.
Программист уронил телефон в унитаз. Вытащил, просушивает. Говорит: главное — не паниковать, бэкап был.
Анекдот про робот-помощника.
Робот-помощник каждое утро делает бутерброд и оставляет записку: 'Съешь меня, а потом — завтрак'.
Анекдот про студента на экзамене.
Студент на экзамене говорит: 'Я всё знаю, просто забыл'. Экзаменатор отвечает: 'Зато я всё помню, даже то, чего вы не знаете'. Студент получил тройку с плюсом.


## Часть 3. Распознаём передачу обратно в текст

Теперь сделаем обратную операцию. STT получает аудиофайл и возвращает гипотезу распознавания. В коротких голосовых сообщениях полезно смотреть не только на итоговый текст, но и на то, где модель могла потерять пунктуацию, имена собственные или редкие слова. Для голосового интерфейса это важно: распознанный текст часто становится входом следующего агента.


In [ ]:
def recognized_text(result) -> str:
    """Преобразует разные варианты ответа SDK в строку."""
    if isinstance(result, str):
        return result
    if hasattr(result, "text"):
        return result.text
    if hasattr(result, "normalized_text"):
        return result.normalized_text
    if hasattr(result, "alternatives") and result.alternatives:
        return result.alternatives[0].text
    if hasattr(result, "__iter__"):
        chunks = []
        for item in result:
            if hasattr(item, "normalized_text"):
                chunks.append(item.normalized_text)
            elif hasattr(item, "text"):
                chunks.append(item.text)
        if chunks:
            return " ".join(chunks)
    return str(result)

def recognize_short_wav(path: str | Path) -> str:
    path = Path(path)
    stt = sdk.speechkit.speech_to_text(
        audio_format="WAV"
    )
    if hasattr(stt, "run_file"):
        result = stt.run_file(path)
    else:
        result = stt.run(path.read_bytes())
    return recognized_text(result)

recognized = recognize_short_wav(radio_path)
printx("### Распознанный текст")
print(recognized)


### Распознанный текст

Приветствую вас с вами минутка анекдотов анекдот про программиста программист уронил телефон в унитаз вытащил просушивает говорит главное не паниковать бэкап был анекдот про робот помощника робот помощник каждое утро делает бутерброд и оставляет записку съешь меня а потом завтрак анекдот про студента на экзамене студент на экзамене говорит я все знаю просто забыл экзаменатор отвечает зато я все помню даже то чего вы не знаете студент получил тройку с плюсом


## Выводы

В этом ноутбуке мы прошли полный цикл короткого голосового сообщения:

1. Получили доступ к SpeechKit через AI Studio SDK.
2. Синтезировали отдельную фразу.
3. Собрали многофрагментный аудиосценарий с разными голосами.
4. Распознали результат обратно в текст.

Такая схема — основа голосовых ботов, аудиотренажёров, голосовых уведомлений и интерфейсов, где пользователь говорит с агентом естественным языком.
